In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
import mlflow
import warnings
warnings.filterwarnings('ignore')

# Load
df = pd.read_csv(r'C:\Users\Karnaveer Singh\Justice_Hq\data\processed\df_model.csv')
print(f"Shape: {df.shape}")
print(f"\nTarget distribution:")
print(df['resolution_bucket'].value_counts().sort_index())
print(f"\nColumns:")
print(df.columns.tolist())

c:\Users\Karnaveer Singh\Justice_Hq\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Shape: (493776, 11)

Target distribution:
resolution_bucket
0_under_6months      158188
1_six_to_24months    185906
2_over_2years        149682
Name: count, dtype: int64

Columns:
['ddl_case_id', 'cino', 'state_code', 'dist_code', 'court_no', 'judge_position', 'type_name', 'filing_year', 'filing_quarter', 'resolution_days', 'resolution_bucket']


In [3]:
import pandas as pd

# Load full dataset for court-history features
df_full = pd.read_csv(r'C:\Users\Karnaveer Singh\Justice_Hq\data\raw\commercial_clean.csv')

# Convert dates
df_full['date_of_filing'] = pd.to_datetime(df_full['date_of_filing'], errors='coerce')
df_full['date_of_decision'] = pd.to_datetime(df_full['date_of_decision'], errors='coerce')

# Keep only resolved cases with valid positive resolution time
df_full_resolved = df_full[df_full['date_of_decision'].notna()].copy()
df_full_resolved['resolution_days'] = (
    df_full_resolved['date_of_decision'] - df_full_resolved['date_of_filing']
).dt.days
df_full_resolved = df_full_resolved[
    (df_full_resolved['resolution_days'] > 0) &
    (df_full_resolved['date_of_filing'].notna())
].copy()

df_full_resolved = df_full_resolved.sort_values(['court_no', 'date_of_filing']).reset_index(drop=True)
print(f'Full resolved rows: {len(df_full_resolved):,}')
print(df_full_resolved[['court_no', 'date_of_filing', 'date_of_decision', 'resolution_days']].head())

Full resolved rows: 2,203,465
   court_no date_of_filing date_of_decision  resolution_days
0         1     2010-01-01       2010-07-15              195
1         1     2010-01-01       2016-04-06             2287
2         1     2010-01-01       2010-11-16              319
3         1     2010-01-01       2010-01-11               10
4         1     2010-01-01       2011-04-30              484


In [ ]:
# Build lagged court-history features and merge them onto df
df_base = pd.read_csv(r'C:\Users\Karnaveer Singh\Justice_Hq\data\processed\df_model.csv')

court_history = df_full_resolved[['ddl_case_id', 'court_no', 'date_of_filing', 'resolution_days']].copy()
court_history = court_history.sort_values(['court_no', 'date_of_filing', 'ddl_case_id'])

court_history['court_historical_median_resolution'] = (
    court_history.groupby('court_no')['resolution_days']
    .transform(lambda s: s.shift().expanding().median())
)

court_history['pending_cases_count'] = court_history.groupby('court_no').cumcount()

global_median_resolution = court_history['resolution_days'].median()
court_history['court_historical_median_resolution'] = court_history['court_historical_median_resolution'].fillna(global_median_resolution)

court_history_features = court_history[['ddl_case_id', 'court_historical_median_resolution', 'pending_cases_count']].copy()
df = df_base.merge(court_history_features, on='ddl_case_id', how='left')
df['court_historical_median_resolution'] = df['court_historical_median_resolution'].fillna(global_median_resolution)
df['pending_cases_count'] = df['pending_cases_count'].fillna(0)

print(f'Base df shape: {df_base.shape}')
print(f'Enriched df shape: {df.shape}')
print(df[['court_historical_median_resolution', 'pending_cases_count']].head())

Base df shape: (493776, 11)
Enriched df shape: (493776, 13)
   court_historical_median_resolution  pending_cases_count
0                               700.5                 1270
1                               662.0                10633
2                               690.0                 1802
3                               670.5                 9936
4                               661.0                 4591


In [15]:
df.head()

,ddl_case_id,cino,state_code,dist_code,court_no,judge_position,type_name,filing_year,filing_quarter,resolution_days,resolution_bucket,court_historical_median_resolution,pending_cases_count,resolution_bucket_v3
0,01-03-02-207401000012010,MHJG040004222010,1,3,2,district and sessions court,4784.0,2010,1,412,1_six_to_24months,700.5,1270,1_over_6months
1,01-03-02-207401000012011,MHJG040039232010,1,3,2,district and sessions court,4784.0,2010,4,565,1_six_to_24months,662.0,10633,1_over_6months
2,01-03-02-207401000022010,MHJG040006132010,1,3,2,district and sessions court,4784.0,2010,1,846,2_over_2years,690.0,1802,1_over_6months
3,01-03-02-207401000022011,MHJG040036672010,1,3,2,district and sessions court,4784.0,2010,4,744,2_over_2years,670.5,9936,1_over_6months
4,01-03-02-207401000032010,MHJG040016072010,1,3,2,district and sessions court,4784.0,2010,2,207,1_six_to_24months,661.0,4591,1_over_6months


In [5]:
# Features
FEATURE_COLS = [
    'state_code', 'dist_code', 'court_no',
    'judge_position', 'type_name',
    'filing_year', 'filing_quarter',
    'court_historical_median_resolution', 'pending_cases_count'
]

# Binary target (y1)
df['binary_target'] = pd.cut(
    df['resolution_days'],
    bins=[-np.inf, 180, np.inf],
    labels=['0_under_6months', '1_over_6months']
)

# 3-class target (y2)
df['bucket_3class'] = pd.cut(
    df['resolution_days'],
    bins=[-np.inf, 180, 730, np.inf],
    labels=['0_under_6months', '1_six_to_24months', '2_over_2years']
)

# Distributions
print("Binary target (y1):")
print(df['binary_target'].value_counts().sort_index())
print(df['binary_target'].value_counts(normalize=True).mul(100).round(1).sort_index())

print("\n3-class target (y2):")
print(df['bucket_3class'].value_counts().sort_index())
print(df['bucket_3class'].value_counts(normalize=True).mul(100).round(1).sort_index())

# Prepare X
X = df[FEATURE_COLS].copy()

categorical_cols = ['state_code', 'dist_code', 'court_no', 'judge_position', 'type_name']
numeric_cols = ['filing_year', 'filing_quarter',
                'court_historical_median_resolution', 'pending_cases_count']

encoders = {}
for col in categorical_cols:
    enc = LabelEncoder()
    X[col] = enc.fit_transform(X[col].astype(str))
    encoders[col] = enc

for col in numeric_cols:
    X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)

# Encode y1 and y2
le_y1 = LabelEncoder()
le_y2 = LabelEncoder()

y1 = le_y1.fit_transform(df['binary_target'].dropna())
y2 = le_y2.fit_transform(df['bucket_3class'].dropna())

print(f"\ny1 classes: {le_y1.classes_}")
print(f"y2 classes: {le_y2.classes_}")
print(f"\nX shape: {X.shape}")

Binary target (y1):
binary_target
0_under_6months    158188
1_over_6months     335588
Name: count, dtype: int64
binary_target
0_under_6months    32.0
1_over_6months     68.0
Name: proportion, dtype: float64

3-class target (y2):
bucket_3class
0_under_6months      158188
1_six_to_24months    185906
2_over_2years        149682
Name: count, dtype: int64
bucket_3class
0_under_6months      32.0
1_six_to_24months    37.6
2_over_2years        30.3
Name: proportion, dtype: float64

y1 classes: ['0_under_6months' '1_over_6months']
y2 classes: ['0_under_6months' '1_six_to_24months' '2_over_2years']

X shape: (493776, 9)


In [26]:
# print("Training XGBoost...")
# xgb_model = xgb.XGBClassifier(
#     n_estimators=300,
#     max_depth=6,
#     learning_rate=0.1,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     random_state=42,
#     eval_metric='mlogloss',
#     verbosity=0
# )

# xgb_model.fit(X_train, y_train)
# xgb_preds = xgb_model.predict(X_test)
# xgb_acc = evaluate_model("XGBoost", y_test, xgb_preds)

In [27]:
# from sklearn.dummy import DummyClassifier

# dummy = DummyClassifier(strategy='most_frequent')
# dummy.fit(X_train, y_train)
# dummy_preds = dummy.predict(X_test)
# dummy_acc = evaluate_model("Dummy Baseline", y_test, dummy_preds)

In [28]:
# feat_importance = pd.DataFrame({
#     'feature': FEATURE_COLS,
#     'importance': xgb_model.feature_importances_
# }).sort_values('importance', ascending=False)

# print(feat_importance)

Type of case is the single biggest predictor of resolution time. That makes legal sense — a cheque bounce case (NI Act Section 138) has a defined fast-track process, while a title dispute can drag on for decades.
State matters more than district — judicial culture at the state level (Orissa vs Himachal Pradesh) dominates over individual district variation.
Filing year has real signal — courts did get faster over 2010-2015 as commercial courts were established.

In [29]:
# from sklearn.ensemble import RandomForestClassifier
# from catboost import CatBoostClassifier

# models = {
#     'XGBoost': xgb.XGBClassifier(n_estimators=300, max_depth=6, 
#                                    learning_rate=0.1, random_state=42, 
#                                    verbosity=0, eval_metric='mlogloss'),
#     'LightGBM': lgb.LGBMClassifier(n_estimators=300, max_depth=6, 
#                                     learning_rate=0.1, random_state=42, 
#                                     verbosity=-1),
#     'CatBoost': CatBoostClassifier(iterations=300, depth=6, 
#                                     learning_rate=0.1, random_state=42, 
#                                     verbose=0),
#     'RandomForest': RandomForestClassifier(n_estimators=300, max_depth=6, 
#                                             random_state=42, n_jobs=-1)
# }

# results = {}
# for name, model in models.items():
#     print(f"Training {name}...")
#     model.fit(X_train, y_train)
#     preds = model.predict(X_test)
#     acc = evaluate_model(name, y_test, preds)
#     results[name] = acc

# print("\n=== FINAL COMPARISON ===")
# for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
#     print(f"{name:15} {acc*100:.2f}%")

In [6]:
def evaluate_model(model_name, y_true, y_pred, le):
    acc = accuracy_score(y_true, y_pred)
    print(f"\n{'='*50}")
    print(f"Model: {model_name}")
    print(f"Accuracy: {acc*100:.2f}%")
    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=le.classes_))
    return acc

# Split
X_train, X_test, y1_train, y1_test = train_test_split(
    X, y1, test_size=0.2, random_state=42, stratify=y1
)
_, _, y2_train, y2_test = train_test_split(
    X, y2, test_size=0.2, random_state=42, stratify=y2
)
print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

best_xgb_params = {
    'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 5,
    'max_depth': 10, 'learning_rate': 0.05, 'colsample_bytree': 0.8
}

# Model 1 — Binary
print("Training Model 1 (Binary)...")
m1 = xgb.XGBClassifier(random_state=42, eval_metric='mlogloss',
                        verbosity=0, n_jobs=-1, **best_xgb_params)
m1.fit(X_train, y1_train)
m1_preds = m1.predict(X_test)
m1_acc = evaluate_model("Model 1 - Binary", y1_test, m1_preds, le_y1)

cm1 = confusion_matrix(y1_test, m1_preds)
cm1_df = pd.DataFrame(cm1, index=le_y1.classes_,
                      columns=[f'Pred: {c}' for c in le_y1.classes_])
print("\nConfusion Matrix - Model 1:")
print(cm1_df)
print("\nRow percentages:")
print(cm1_df.div(cm1_df.sum(axis=1), axis=0).mul(100).round(1))

# Model 2 — 3-class
print("\nTraining Model 2 (3-class)...")
m2 = xgb.XGBClassifier(random_state=42, eval_metric='mlogloss',
                        verbosity=0, n_jobs=-1, **best_xgb_params)
m2.fit(X_train, y2_train)
m2_preds = m2.predict(X_test)
m2_acc = evaluate_model("Model 2 - 3 class", y2_test, m2_preds, le_y2)

cm2 = confusion_matrix(y2_test, m2_preds)
cm2_df = pd.DataFrame(cm2, index=le_y2.classes_,
                      columns=[f'Pred: {c}' for c in le_y2.classes_])
print("\nConfusion Matrix - Model 2:")
print(cm2_df)
print("\nRow percentages:")
print(cm2_df.div(cm2_df.sum(axis=1), axis=0).mul(100).round(1))

Train: 395,020 | Test: 98,756
Training Model 1 (Binary)...

Model: Model 1 - Binary
Accuracy: 81.35%

Classification Report:
                 precision    recall  f1-score   support

0_under_6months       0.82      0.54      0.65     31638
 1_over_6months       0.81      0.94      0.87     67118

       accuracy                           0.81     98756
      macro avg       0.81      0.74      0.76     98756
   weighted avg       0.81      0.81      0.80     98756


Confusion Matrix - Model 1:
                 Pred: 0_under_6months  Pred: 1_over_6months
0_under_6months                  17078                 14560
1_over_6months                    3856                 63262

Row percentages:
                 Pred: 0_under_6months  Pred: 1_over_6months
0_under_6months                   54.0                  46.0
1_over_6months                     5.7                  94.3

Training Model 2 (3-class)...

Model: Model 2 - 3 class
Accuracy: 36.01%

Classification Report:
                   

In [9]:
# Stage 2 hybrid training: only delayed cases (> 6 months) from the training split
stage2_label_map = {'1_six_to_24months': 0, '2_over_2years': 1}

train_frame = df.loc[X_train.index, ['binary_target', 'bucket_3class']].copy()
long_cases_mask_train = train_frame['binary_target'].eq('1_over_6months')

X_train_stage2 = X_train.loc[long_cases_mask_train].copy()
y2_train_binary = train_frame.loc[long_cases_mask_train, 'bucket_3class'].map(stage2_label_map).astype(int)

print(f"Training focused Stage 2 Model on {X_train_stage2.shape[0]:,} delayed cases...")
print(f"Stage 2 target distribution: {np.bincount(y2_train_binary)}")

m2_binary = xgb.XGBClassifier(**best_xgb_params, eval_metric='logloss', random_state=42, verbosity=0, n_jobs=-1)
m2_binary.fit(X_train_stage2, y2_train_binary)

# Optional quick sanity check on the held-out set, restricted to delayed cases only
test_frame = df.loc[X_test.index, ['binary_target', 'bucket_3class']].copy()
long_cases_mask_test = test_frame['binary_target'].eq('1_over_6months')
X_test_stage2 = X_test.loc[long_cases_mask_test].copy()
y2_test_binary = test_frame.loc[long_cases_mask_test, 'bucket_3class'].map(stage2_label_map).astype(int)

m2_binary_preds = m2_binary.predict(X_test_stage2)
print(f"Stage 2 test cases: {X_test_stage2.shape[0]:,}")
print(f"Stage 2 accuracy: {accuracy_score(y2_test_binary, m2_binary_preds) * 100:.2f}%")
print(classification_report(y2_test_binary, m2_binary_preds, target_names=['1_six_to_24months', '2_over_2years']))

cm2_binary = confusion_matrix(y2_test_binary, m2_binary_preds)
cm2_binary_df = pd.DataFrame(
    cm2_binary,
    index=['Actual: 1_six_to_24months', 'Actual: 2_over_2years'],
    columns=['Pred: 1_six_to_24months', 'Pred: 2_over_2years']
)
print("\nStage 2 Confusion Matrix:")
print(cm2_binary_df)
print("\nRow percentages:")
print(cm2_binary_df.div(cm2_binary_df.sum(axis=1), axis=0).mul(100).round(1))

Training focused Stage 2 Model on 268,470 delayed cases...
Stage 2 target distribution: [148617 119853]
Stage 2 test cases: 67,118
Stage 2 accuracy: 66.34%
                   precision    recall  f1-score   support

1_six_to_24months       0.67      0.78      0.72     37289
    2_over_2years       0.65      0.52      0.58     29829

         accuracy                           0.66     67118
        macro avg       0.66      0.65      0.65     67118
     weighted avg       0.66      0.66      0.66     67118


Stage 2 Confusion Matrix:
                           Pred: 1_six_to_24months  Pred: 2_over_2years
Actual: 1_six_to_24months                    28922                 8367
Actual: 2_over_2years                        14226                15603

Row percentages:
                           Pred: 1_six_to_24months  Pred: 2_over_2years
Actual: 1_six_to_24months                     77.6                 22.4
Actual: 2_over_2years                         47.7                 52.3


In [11]:
tuned_stage2_params = {
    # 1. Combat Imbalance
    'scale_pos_weight': 148617 / 119853,

    # 2. Control Overfitting
    'max_depth': 8,
    'min_child_weight': 10,

    # 3. Learning Speed and Coverage
    'n_estimators': 600,
    'learning_rate': 0.03,
    'subsample': 0.8,
    'colsample_bytree': 0.8
}

print('Training tuned Stage 2 model on delayed cases only...')
print(f"Tuned Stage 2 params: {tuned_stage2_params}")

m2_binary_tuned = xgb.XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    verbosity=0,
    n_jobs=-1,
    **tuned_stage2_params
)

m2_binary_tuned.fit(X_train_stage2, y2_train_binary)

tuned_stage2_preds = m2_binary_tuned.predict(X_test_stage2)
print(f"Tuned Stage 2 test cases: {X_test_stage2.shape[0]:,}")
print(f"Tuned Stage 2 accuracy: {accuracy_score(y2_test_binary, tuned_stage2_preds) * 100:.2f}%")
print(classification_report(y2_test_binary, tuned_stage2_preds, target_names=['1_six_to_24months', '2_over_2years']))

cm2_binary_tuned = confusion_matrix(y2_test_binary, tuned_stage2_preds)
cm2_binary_tuned_df = pd.DataFrame(
    cm2_binary_tuned,
    index=['Actual: 1_six_to_24months', 'Actual: 2_over_2years'],
    columns=['Pred: 1_six_to_24months', 'Pred: 2_over_2years']
)
print('\nTuned Stage 2 Confusion Matrix:')
print(cm2_binary_tuned_df)
print('\nRow percentages:')
print(cm2_binary_tuned_df.div(cm2_binary_tuned_df.sum(axis=1), axis=0).mul(100).round(1))

Training tuned Stage 2 model on delayed cases only...
Tuned Stage 2 params: {'scale_pos_weight': 1.2399939926409853, 'max_depth': 8, 'min_child_weight': 10, 'n_estimators': 600, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.8}
Tuned Stage 2 test cases: 67,118
Tuned Stage 2 accuracy: 65.14%
                   precision    recall  f1-score   support

1_six_to_24months       0.70      0.65      0.68     37289
    2_over_2years       0.60      0.65      0.62     29829

         accuracy                           0.65     67118
        macro avg       0.65      0.65      0.65     67118
     weighted avg       0.66      0.65      0.65     67118


Tuned Stage 2 Confusion Matrix:
                           Pred: 1_six_to_24months  Pred: 2_over_2years
Actual: 1_six_to_24months                    24365                12924
Actual: 2_over_2years                        10474                19355

Row percentages:
                           Pred: 1_six_to_24months  Pred: 2_over_2ye

In [12]:
import pandas as pd

# Feature importance for the final gatekeeper and tuned Stage 2 models
m1_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Gatekeeper_Importance': m1.feature_importances_
}).sort_values(by='Gatekeeper_Importance', ascending=False)

m2_importance = pd.DataFrame({
    'Feature': X_train_stage2.columns,
    'Risk_Importance': m2_binary_tuned.feature_importances_
}).sort_values(by='Risk_Importance', ascending=False)

print('--- Top Drivers for Case Delay (Model 1 Gatekeeper) ---')
print(m1_importance.head(5))

print('\n--- Top Drivers for Severe 2+ Year Bottlenecks (Model 2 Risk) ---')
print(m2_importance.head(5))

--- Top Drivers for Case Delay (Model 1 Gatekeeper) ---
          Feature  Gatekeeper_Importance
4       type_name               0.444118
5     filing_year               0.122743
0      state_code               0.107355
6  filing_quarter               0.087712
2        court_no               0.064065

--- Top Drivers for Severe 2+ Year Bottlenecks (Model 2 Risk) ---
       Feature  Risk_Importance
5  filing_year         0.402166
0   state_code         0.140487
4    type_name         0.108037
2     court_no         0.081922
1    dist_code         0.072711


In [13]:
import joblib
import os

os.makedirs(r'C:\Users\Karnaveer Singh\Justice_Hq\models', exist_ok=True)

joblib.dump(m1, r'C:\Users\Karnaveer Singh\Justice_Hq\models\stage1_binary.pkl')
joblib.dump(m2_binary, r'C:\Users\Karnaveer Singh\Justice_Hq\models\stage2_medium_long.pkl')
joblib.dump(le_y1, r'C:\Users\Karnaveer Singh\Justice_Hq\models\le_stage1.pkl')
joblib.dump(le_y2, r'C:\Users\Karnaveer Singh\Justice_Hq\models\le_stage2.pkl')
joblib.dump(encoders, r'C:\Users\Karnaveer Singh\Justice_Hq\models\feature_encoders.pkl')

print('All models saved.')
print('\nFinal model summary:')
print(f'Stage 1 (fast vs slow):      {m1_acc*100:.2f}%')
print(f'Stage 2 (medium vs long):    {m2_acc*100:.2f}%')

All models saved.

Final model summary:
Stage 1 (fast vs slow):      81.35%
Stage 2 (medium vs long):    36.01%


from pathlib import Path

# 1. Setup path and create the folder if it's missing
output_file = Path("processed/df_model.csv")
output_file.parent.mkdir(parents=True, exist_ok=True)

# 2. Save the DataFrame directly
df.to_csv(output_file, index=False, compression="gzip")
print(f"SUCCESS! Saved to {output_file}")

In [18]:
from pathlib import Path

# 1. Setup path and create the folder if it's missing
output_file = Path("processed/df_model.csv")
output_file.parent.mkdir(parents=True, exist_ok=True)

# 2. Save the DataFrame directly
df.to_csv(r'C:\Users\Karnaveer Singh\Justice_Hq\data\processed\df_model.csv', index=False)
print(f"SUCCESS! Saved to {output_file}")
print(df.columns)

SUCCESS! Saved to processed\df_model.csv
Index(['ddl_case_id', 'cino', 'state_code', 'dist_code', 'court_no',
       'judge_position', 'type_name', 'filing_year', 'filing_quarter',
       'resolution_days', 'resolution_bucket',
       'court_historical_median_resolution', 'pending_cases_count',
       'binary_target', 'bucket_3class'],
      dtype='object')


In [15]:
df.head()

,ddl_case_id,cino,state_code,dist_code,court_no,judge_position,type_name,filing_year,filing_quarter,resolution_days,resolution_bucket,court_historical_median_resolution,pending_cases_count,binary_target,bucket_3class
0,01-03-02-207401000012010,MHJG040004222010,1,3,2,district and sessions court,4784.0,2010,1,412,1_six_to_24months,700.5,1270,1_over_6months,1_six_to_24months
1,01-03-02-207401000012011,MHJG040039232010,1,3,2,district and sessions court,4784.0,2010,4,565,1_six_to_24months,662.0,10633,1_over_6months,1_six_to_24months
2,01-03-02-207401000022010,MHJG040006132010,1,3,2,district and sessions court,4784.0,2010,1,846,2_over_2years,690.0,1802,1_over_6months,2_over_2years
3,01-03-02-207401000022011,MHJG040036672010,1,3,2,district and sessions court,4784.0,2010,4,744,2_over_2years,670.5,9936,1_over_6months,2_over_2years
4,01-03-02-207401000032010,MHJG040016072010,1,3,2,district and sessions court,4784.0,2010,2,207,1_six_to_24months,661.0,4591,1_over_6months,1_six_to_24months


In [19]:
df_full = pd.read_csv(r'C:\Users\Karnaveer Singh\Justice_Hq\data\raw\commercial_clean.csv')

# Only use resolved cases for median calculation
df_full_resolved = df_full[df_full['resolution_days'].notna()].copy()
df_full_resolved = df_full_resolved[
    (df_full_resolved['resolution_days'] > 0) & 
    (df_full_resolved['resolution_days'] <= 10950)
]

# Precompute per-court stats
court_lookup = df_full_resolved.groupby('court_no').agg(
    court_historical_median_resolution=('resolution_days', 'median'),
    pending_cases_count=('resolution_days', 'count')  # count as proxy for caseload
).reset_index()

global_median = df_full_resolved['resolution_days'].median()
global_pending = court_lookup['pending_cases_count'].median()

print(f"Courts in lookup: {len(court_lookup)}")
print(f"Global median: {global_median:.0f} days")
print(court_lookup.describe())

court_lookup.to_csv(r'C:\Users\Karnaveer Singh\Justice_Hq\data\lookups\court_lookup.csv', index=False)
print("Saved.")

Courts in lookup: 49
Global median: 293 days
        court_no  court_historical_median_resolution  pending_cases_count
count  49.000000                           49.000000            49.000000
mean   25.530612                          486.642857         44968.673469
std    15.265842                          467.052203         71914.333259
min     1.000000                          130.000000             2.000000
25%    13.000000                          298.000000          1730.000000
50%    25.000000                          368.000000         16039.000000
75%    37.000000                          465.000000         57323.000000
max    59.000000                         2560.000000        378924.000000
Saved.


In [21]:
court_lookup = court_history.merge(
    df_full_resolved[['ddl_case_id', 'state_code', 'dist_code']], 
    on='ddl_case_id', how='left'
)

court_lookup = court_lookup.groupby(['state_code', 'dist_code', 'court_no']).agg(
    court_historical_median_resolution=('court_historical_median_resolution', 'last'),
    pending_cases_count=('pending_cases_count', 'last')
).reset_index()

court_lookup['court_historical_median_resolution'] = court_lookup[
    'court_historical_median_resolution'].fillna(global_median)
court_lookup['pending_cases_count'] = court_lookup[
    'pending_cases_count'].fillna(global_pending)

print(f"Unique courts in lookup: {len(court_lookup):,}")
print(court_lookup.head(10))

court_lookup.to_csv(
    r'C:\Users\Karnaveer Singh\Justice_Hq\data\lookups\court_lookup.csv',
    index=False
)
print("Saved.")

Unique courts in lookup: 3,906
   state_code  dist_code  court_no  court_historical_median_resolution  \
0           1          1         3                               522.0   
1           1          1         7                              1088.0   
2           1          1        10                               357.0   
3           1          2         1                               220.0   
4           1          2         3                               616.0   
5           1          3         2                               656.0   
6           1          3         4                               556.0   
7           1          3         5                              1186.0   
8           1          3        20                               900.0   
9           1          4         1                               177.0   

   pending_cases_count  
0               210341  
1                10056  
2                55843  
3               152475  
4                40670  
5   

In [22]:
import mlflow
import os

# Set DagsHub as your MLflow tracking server
os.environ['MLFLOW_TRACKING_USERNAME'] = 'Karnaveer10'
os.environ['MLFLOW_TRACKING_PASSWORD'] = '775986321f759a9daf8f7e20bfc2217b4818155a'  # get this from DagsHub

mlflow.set_tracking_uri('https://dagshub.com/Karnaveer10/JustIt.mlflow')
mlflow.set_experiment('justiceiq-models')

print("MLflow connected to DagsHub")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

2026/06/11 12:32:44 INFO mlflow.tracking.fluent: Experiment with name 'justiceiq-models' does not exist. Creating a new experiment.


MLflow connected to DagsHub
Tracking URI: https://dagshub.com/Karnaveer10/JustIt.mlflow


In [23]:
df.columns

Index(['ddl_case_id', 'cino', 'state_code', 'dist_code', 'court_no',
       'judge_position', 'type_name', 'filing_year', 'filing_quarter',
       'resolution_days', 'resolution_bucket',
       'court_historical_median_resolution', 'pending_cases_count',
       'binary_target', 'bucket_3class'],
      dtype='object')

In [ ]:
import mlflow
import mlflow.xgboost
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
import lightgbm as lgb

mlflow.set_tracking_uri('https://dagshub.com/Karnaveer10/JustIt.mlflow')
mlflow.set_experiment('justiceiq-models')

FEATURE_COLS = [
    'state_code', 'dist_code', 'court_no',
    'judge_position', 'type_name',
    'filing_year', 'filing_quarter',
    'court_historical_median_resolution', 'pending_cases_count'
]

best_params = {
    'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 5,
    'max_depth': 10, 'learning_rate': 0.05, 'colsample_bytree': 0.8
}

# ── Prepare X, y1, y2 ─────────────────────────────────────
df = pd.read_csv(r'C:\Users\Karnaveer Singh\Justice_Hq\data\processed\df_model.csv')

X = df[FEATURE_COLS].copy()
encoders = {}
for col in ['state_code', 'dist_code', 'court_no', 'judge_position', 'type_name']:
    enc = LabelEncoder()
    X[col] = enc.fit_transform(X[col].astype(str))
    encoders[col] = enc
for col in ['filing_year', 'filing_quarter', 
            'court_historical_median_resolution', 'pending_cases_count']:
    X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)

le_y1 = LabelEncoder()
le_y2 = LabelEncoder()
le_y3 = LabelEncoder()

y1 = le_y1.fit_transform(df['binary_target'].dropna())
y2 = le_y2.fit_transform(df['bucket_3class'].dropna())
y3 = le_y3.fit_transform(df['resolution_bucket'].dropna())

X_train, X_test, y1_train, y1_test = train_test_split(
    X, y1, test_size=0.2, random_state=42, stratify=y1)
_, _, y2_train, y2_test = train_test_split(
    X, y2, test_size=0.2, random_state=42, stratify=y2)
_, _, y3_train, y3_test = train_test_split(
    X, y3, test_size=0.2, random_state=42, stratify=y3)

# Stage 2 — slow cases only
slow_mask_train = y1_train == 1
slow_mask_test = y1_test == 1
X_train_s2 = X_train[slow_mask_train]
X_test_s2 = X_test[slow_mask_test]
y2_train_s2 = y3_train[slow_mask_train]
y2_test_s2 = y3_test[slow_mask_test]

print("Data prepared. Starting MLflow logging...")

# ── Helper ────────────────────────────────────────────────
def log_run(run_name, model, X_tr, y_tr, X_te, y_te, le, params, tags={}):
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags(tags)
        mlflow.log_params(params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        acc = accuracy_score(y_te, preds)
        report = classification_report(y_te, preds,
                    target_names=le.classes_, output_dict=True)
        mlflow.log_metric("accuracy", round(acc, 4))
        mlflow.log_metric("macro_f1", round(report['macro avg']['f1-score'], 4))
        mlflow.log_metric("macro_precision", round(report['macro avg']['precision'], 4))
        mlflow.log_metric("macro_recall", round(report['macro avg']['recall'], 4))
        mlflow.xgboost.log_model(model, run_name)
        print(f"✓ {run_name}: {acc*100:.2f}%")
        return model, acc

# ── Run 1: Dummy Baseline ─────────────────────────────────
with mlflow.start_run(run_name="dummy_baseline"):
    mlflow.set_tag("stage", "baseline")
    dummy = DummyClassifier(strategy='most_frequent')
    dummy.fit(X_train, y1_train)
    acc = accuracy_score(y1_test, dummy.predict(X_test))
    mlflow.log_param("strategy", "most_frequent")
    mlflow.log_metric("accuracy", round(acc, 4))
    print(f"✓ dummy_baseline: {acc*100:.2f}%")

# ── Run 2: XGBoost 3-class default ───────────────────────
log_run("xgboost_3class_default",
    xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                      random_state=42, eval_metric='mlogloss', verbosity=0),
    X_train, y2_train, X_test, y2_test, le_y2,
    {'n_estimators':300,'max_depth':6,'learning_rate':0.1,'target':'3class'},
    {'stage':'exploration','target':'3class'})

# ── Run 3: LightGBM 3-class ───────────────────────────────
with mlflow.start_run(run_name="lightgbm_3class"):
    mlflow.set_tag("stage", "exploration")
    lgb_m = lgb.LGBMClassifier(n_estimators=300, max_depth=6,
                                learning_rate=0.1, random_state=42, verbosity=-1)
    lgb_m.fit(X_train, y2_train)
    preds = lgb_m.predict(X_test)
    acc = accuracy_score(y2_test, preds)
    mlflow.log_params({'n_estimators':300,'max_depth':6,'learning_rate':0.1})
    mlflow.log_metric("accuracy", round(acc, 4))
    print(f"✓ lightgbm_3class: {acc*100:.2f}%")

# ── Run 4: XGBoost 3-class tuned ─────────────────────────
log_run("xgboost_3class_tuned",
    xgb.XGBClassifier(random_state=42, eval_metric='mlogloss', 
                      verbosity=0, **best_params),
    X_train, y2_train, X_test, y2_test, le_y2,
    {**best_params, 'target':'3class'},
    {'stage':'tuned','target':'3class'})

# ── Run 5: XGBoost binary stage1 ─────────────────────────
log_run("xgboost_binary_stage1",
    xgb.XGBClassifier(random_state=42, eval_metric='logloss',
                      verbosity=0, **best_params),
    X_train, y1_train, X_test, y1_test, le_y1,
    {**best_params, 'target':'binary'},
    {'stage':'final','target':'binary','model':'stage1'})

# ── Run 6: XGBoost stage2 medium vs long ─────────────────
log_run("xgboost_stage2_medium_long",
    xgb.XGBClassifier(random_state=42, eval_metric='logloss',
                      verbosity=0, **best_params),
    X_train_s2, y2_train_s2, X_test_s2, y2_test_s2, le_y3,
    {**best_params, 'target':'medium_vs_long'},
    {'stage':'final','target':'medium_vs_long','model':'stage2'})

print("\n✅ All runs logged.")
print("View at: https://dagshub.com/Karnaveer10/JustIt.mlflow")

c:\Users\Karnaveer Singh\Justice_Hq\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MlflowException: API request to endpoint was successful but the response body was not in a valid JSON format. Response body: '<!DOCTYPE html><html class="__variable_13fb82"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1"/><link rel="preload" href="/_next/static/media/3703c28dcda155b1-s.p.woff2" as="font" crossorigin="" type="font/woff2"/><link rel="preload" href="/_next/static/media/e4af272ccee01ff0-s.p.woff2" as="font" crossorigin="" type="font/woff2"/><link rel="preload" href="/next-public/img/logos/dagshub-icon.svg" as="image"/><link rel="stylesheet" href="/_next/static/css/7e7d96b1e6991756.css" data-precedence="next"/><link rel="stylesheet" href="/_next/static/css/cc8262f9be461d8b.css" data-precedence="next"/><link rel="stylesheet" href="/_next/static/css/518f0eeb27253b9a.css" data-precedence="next"/><link rel="stylesheet" href="/_next/static/css/4b72dbfbaf1b2e49.css" data-precedence="next"/><link rel="stylesheet" href="/_next/static

In [ ]:
df_model.head()